# Lab 1 - Exercise 3.2: Retrieval as Training-free Classification

Questo notebook tratta GTSRB come un problema di retrieval: il training set diventa una gallery di feature, il test set diventa un insieme di query. Non facciamo fine-tuning; valutiamo quanto sono utili le rappresentazioni di backbone pre-addestrati.

Il notebook usa cache di feature generate dalla pipeline nuova in `artifacts/features/`. Le metriche devono essere rigenerate da queste cache, non copiate dal notebook storico.


---
### Exercise 3.2: Retrieval as Training-free Classification

In questo esercizio trattiamo il training set come **gallery** e il test set come insieme di **query**. Ogni immagine viene trasformata in un vettore di feature tramite un backbone pre-addestrato.

La pipeline e' composta da tre parti:

1. estrazione generica delle feature da un modello e un DataLoader;
2. ricerca nella gallery tramite similarita' tra feature;
3. valutazione retrieval e classificazione training-free con Nearest-Mean Classifier.

Non alleniamo il modello: usiamo direttamente le rappresentazioni apprese su ImageNet.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import classification_report
from torchvision.datasets import GTSRB

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import load_config
from dla_lab1.data import build_retrieval_dataloaders
from dla_lab1.features import (
    compare_similarity_precision_at_k,
    cosine_similarity_matrix,
    extract_features,
    load_feature_cache,
    nearest_mean_classifier,
    retrieval_mean_average_precision,
    retrieval_mean_average_precision_chunked,
    retrieval_precision_at_k,
    save_feature_cache,
)
from dla_lab1.models import build_feature_extractor, count_parameters
from dla_lab1.paths import resolve_path
from dla_lab1.visualize import plot_retrieval_results

config = load_config(ROOT / "config" / "config.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_EXTRACTION = False
RUN_RETRIEVAL_EVAL = True
RUN_SIMILARITY_COMPARISON = True
RUN_VISUALIZATION = False
BACKBONE = "resnet50"

print(f"Project root: {ROOT}")
print(f"Device: {device}")


## Funzioni usate

Le funzioni lunghe non sono definite nel notebook, ma nei file `.py` dentro `src/dla_lab1`:

- `build_retrieval_dataloaders`: crea gallery e query senza augmentation;
- `build_feature_extractor`: carica il backbone pre-addestrato e rimuove la testa finale;
- `extract_features`: passa le immagini nel backbone e salva i vettori su CPU;
- `cosine_similarity_matrix`: calcola la similarita' coseno in modo vettorizzato;
- `retrieval_precision_at_k`: valuta quante immagini corrette compaiono nelle top-K;
- `retrieval_mean_average_precision`: calcola mAP globale e AP media per classe;
- `nearest_mean_classifier`: implementa il classificatore training-free basato sui centroidi.

Nei file `.py` ogni funzione ha una nota che spiega a cosa serve.

In [ ]:
data_root = resolve_path(config["paths"]["data_root"], ROOT)
image_size = int(config["dataset"]["image_size"])
batch_size = int(config["hardware"]["batch_size_feature_extraction"])
num_workers = int(config["dataset"].get("num_workers", 2))
pin_memory = bool(config["dataset"].get("pin_memory", True))

retrieval_loaders = build_retrieval_dataloaders(
    data_root=data_root,
    image_size=image_size,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

loader_summary = pd.DataFrame([
    ["gallery", len(retrieval_loaders["gallery"].dataset), batch_size],
    ["query", len(retrieval_loaders["query"].dataset), batch_size],
], columns=["Split", "Images", "Batch size"])

loader_summary

## Feature extraction

La gallery contiene le feature del training set, mentre le query contengono le feature del test set. La cella seguente e' opzionale perche' l'estrazione puo' richiedere diversi minuti, soprattutto con backbone piu' grandi.

La pipeline e' generica: cambiando `BACKBONE` si possono provare modelli diversi, ad esempio `resnet18` e `resnet50`, senza riscrivere il resto del notebook.


In [ ]:
feature_cache = ROOT / "artifacts" / "features" / f"ex3_2_{BACKBONE}_retrieval_features.pt"

if RUN_EXTRACTION:
    feature_extractor = build_feature_extractor(BACKBONE, weights="DEFAULT")
    print(count_parameters(feature_extractor))

    gallery_features, gallery_labels = extract_features(
        feature_extractor,
        retrieval_loaders["gallery"],
        device,
    )
    query_features, query_labels = extract_features(
        feature_extractor,
        retrieval_loaders["query"],
        device,
    )

    save_feature_cache(
        feature_cache,
        gallery_features=gallery_features,
        gallery_labels=gallery_labels,
        query_features=query_features,
        query_labels=query_labels,
    )
    print(f"Feature salvate in: {feature_cache}")
elif feature_cache.exists():
    cache = load_feature_cache(feature_cache)
    gallery_features = cache["gallery_features"]
    gallery_labels = cache["gallery_labels"]
    query_features = cache["query_features"]
    query_labels = cache["query_labels"]
    print(f"Feature caricate da: {feature_cache}")
else:
    print("Feature extraction non eseguita. Imposta RUN_EXTRACTION = True per generare la cache.")


## Confronto tra similarita'

Qui confrontiamo Dot Product, Cosine Similarity ed Euclidean Distance ricalcolando Precision@K dalle feature correnti. La funzione lavora a blocchi sulle query, quindi non tiene in memoria tre matrici complete contemporaneamente.

La scelta della similarita' e' importante: nel retrieval non alleniamo un classificatore, quindi l'ordine dei vicini dipende direttamente da come confrontiamo i vettori di feature.


In [ ]:
if RUN_SIMILARITY_COMPARISON and "gallery_features" in globals():
    recalculated_similarity_metrics = compare_similarity_precision_at_k(
        query_features=query_features,
        gallery_features=gallery_features,
        query_labels=query_labels,
        gallery_labels=gallery_labels,
        k_values=(1, 5, 10),
        chunk_size=256,
    )

    recalculated_similarity_results = (
        pd.DataFrame.from_dict(recalculated_similarity_metrics, orient="index")
        .rename_axis("Similarity")
        .reset_index()
        .rename(columns={1: "Precision@1", 5: "Precision@5", 10: "Precision@10"})
        .sort_values("Precision@1", ascending=False)
    )

    display(recalculated_similarity_results)
else:
    print("Confronto similarity non eseguito: carica/genera prima le feature oppure imposta RUN_SIMILARITY_COMPARISON = True.")


## Analisi delle similarita'

Confrontiamo Dot Product, Cosine Similarity ed Euclidean Distance perche' il retrieval dipende direttamente dalla metrica scelta. La Cosine Similarity e' spesso piu' stabile per feature profonde, perche' confronta la direzione dei vettori e riduce l'effetto della loro norma.

La valutazione viene calcolata a blocchi, cosi' il notebook non deve creare piu' matrici query-gallery complete in memoria. Questo e' importante per GTSRB: il test set ha 12.630 query e la gallery ha 26.640 immagini.


In [ ]:
if RUN_RETRIEVAL_EVAL and "gallery_features" in globals():
    if "recalculated_similarity_metrics" not in globals():
        recalculated_similarity_metrics = compare_similarity_precision_at_k(
            query_features=query_features,
            gallery_features=gallery_features,
            query_labels=query_labels,
            gallery_labels=gallery_labels,
            k_values=(1, 5, 10),
            chunk_size=256,
        )
    precision_at_k = recalculated_similarity_metrics["Cosine similarity"]
    precision_at_k
else:
    print("Valutazione retrieval non eseguita: carica/genera prima le feature.")


## Nearest-Mean Classifier

Il retrieval puro restituisce una lista ordinata di immagini simili, ma non e' direttamente un classificatore. Per ottenere una predizione di classe usiamo il Nearest-Mean Classifier:

1. calcoliamo il centroide delle feature per ciascuna delle 43 classi;
2. confrontiamo ogni query con i 43 centroidi;
3. assegniamo la classe del centroide piu' vicino.

Questo e' training-free: non ci sono gradienti, epoche o optimizer.

In [ ]:
retrieval_backbone_candidates = pd.DataFrame([
    ["resnet18", 512, "backbone leggero, veloce da estrarre"],
    ["resnet50", 2048, "backbone piu' pesante, feature piu' ricche"],
], columns=["Backbone", "Feature dim", "Reason"])

retrieval_backbone_candidates


In [ ]:
if RUN_RETRIEVAL_EVAL and "gallery_features" in globals():
    nmc_predictions = nearest_mean_classifier(
        gallery_features,
        gallery_labels,
        query_features,
    )
    nmc_accuracy = (nmc_predictions == query_labels).float().mean().item()
    nmc_summary = pd.DataFrame([
        [BACKBONE, nmc_accuracy],
    ], columns=["Backbone", "NMC Accuracy"])

    display(nmc_summary)
    print(classification_report(query_labels.numpy(), nmc_predictions.numpy(), zero_division=0))
else:
    print("NMC non eseguito. Imposta RUN_RETRIEVAL_EVAL = True dopo aver caricato o generato le feature.")


## mAP e valutazione per classe

La Average Precision misura la qualita' del ranking retrieval considerando tutte le immagini rilevanti nella gallery. Nel notebook viene calcolata solo quando `RUN_RETRIEVAL_EVAL = True`, cosi' il risultato dipende dalle feature effettivamente caricate o generate nella sessione corrente.

Questa metrica e' diversa dalla NMC accuracy: la AP valuta tutto il ranking query-gallery, mentre NMC confronta ogni query solo con i centroidi delle 43 classi.


In [ ]:
if RUN_RETRIEVAL_EVAL and "gallery_features" in globals():
    map_result = retrieval_mean_average_precision_chunked(
        query_features=query_features,
        gallery_features=gallery_features,
        query_labels=query_labels,
        gallery_labels=gallery_labels,
        num_classes=int(config["project"]["num_classes"]),
        chunk_size=128,
    )
    per_class_ap = pd.DataFrame(
        sorted(map_result["per_class_ap"].items()),
        columns=["class_id", "average_precision"],
    )
    print(f"mAP: {map_result['mAP']:.4f}")
    print(f"macro mAP by class: {map_result['macro_mAP_by_class']:.4f}")
    display(per_class_ap)
else:
    print("mAP non ricalcolata: carica/genera prima le feature.")


## Visualizzazione qualitativa opzionale

Questa parte mostra una query e le top-K immagini recuperate dalla gallery. Non e' necessaria per la metrica finale, ma aiuta a discutere visivamente quando il retrieval funziona e quando confonde segnali simili.


In [ ]:
if RUN_VISUALIZATION and "cosine_scores" in globals():
    raw_gallery = GTSRB(str(data_root), split="train", download=False)
    raw_query = GTSRB(str(data_root), split="test", download=False)
    plot_retrieval_results(raw_query, raw_gallery, cosine_scores, query_index=0, k=5)
else:
    print("Visualizzazione non eseguita. Imposta RUN_VISUALIZATION = True dopo aver calcolato cosine_scores.")


## Conclusione Exercise 3.2

L'esercizio e' coperto dai tre elementi richiesti:

1. una funzione generica di feature extraction, utilizzabile con backbone diversi;
2. una pipeline di query gallery basata su similarita' tra feature;
3. metriche retrieval e classificazione training-free con Nearest-Mean Classifier.

Il punto metodologico principale e' che questo approccio non aggiorna i pesi del backbone. Se le feature pre-addestrate sono buone, classi simili finiscono vicine nello spazio delle rappresentazioni; se non sono abbastanza specifiche per segnali stradali piccoli, retrieval e NMC restano inferiori al fine-tuning. Per questo l'esercizio e' utile come confronto: misura quanto possiamo ottenere senza training e quanto guadagniamo invece adattando `layer4` nell'Exercise 3.1.
